# Talent Intelligence Platform
### Millennium Business Development — Data Science case study

**Jasmine (Yuanzhi) Chen** · August 2026

| Deliverable | Where |
|---|---|
| Live application | **https://m-case-study-jasmine.streamlit.app/** |
| Code repository | https://github.com/yc4379-commits/m-case-study |
| Parsed data (JSON / CSV) | `data/candidates.json` · `data/candidates.csv` |
| This notebook | the walk-through: every section below runs against the committed code and data |

The task: parse ten resumes (PDF and Word) with an LLM API into structured
JSON/CSV, and build a Streamlit application where a Business Development
user can search and filter candidates against job requisitions, with
distribution insights — designed to scale beyond ten resumes.

Three commitments shape every design decision in this system:

1. **Hard constraints disqualify; soft signals rank.** A candidate outside
   the role's region or experience band is not an 82% match — they are not a
   match. Keeping the two separate is what lets this search honestly return
   **zero results**, which commercial matching tools cannot do.
2. **Every claim carries its evidence.** No classification appears without
   the resume sentence that produced it, verified to appear verbatim in the
   source. This is not decoration: the worst scoring bug in this project was
   invisible in the aggregate numbers and obvious the moment one evidence
   quote was read (§7).
3. **Data quality is visible, never silent.** Every record carries a parse
   confidence with the specific reasons, and formatting deliberately carries
   **zero** weight in it — in this corpus, messy formatting tracks regional
   convention, not candidate quality.

## 1 · Architecture

```
data/resumes/*.{docx,pdf}
        │
        ▼  src/extract.py        paragraphs + tables + floating text boxes;
        │                        PDF column separation; ligature repair
        ▼  src/parse.py          schema-constrained LLM extraction (Anthropic
        │                        tool use), validation with corrective retry,
        │                        verbatim-quote verification, content-hash cache
        ▼  src/checks.py         deterministic checks: gaps, overlaps, contact
        │                        details, credential pairs, city-zip sanity
        ▼  src/enrich.py         knowledge-base lookups, tenure arithmetic,
        │                        entity resolution, human flag-triage,
        │                        confidence scoring
        ▼
data/candidates.json ──► app.py (Streamlit) ──► src/match.py (requisitions)
                                    │
                                    └──► src/evaluate.py (accuracy vs human labels)
```

The split of labour is a design position: **the model is asked only for
judgement** (fundamental vs systematic, which sectors, is this an investment
role — each with evidence), and **everything code can settle is settled by
code** (tenure arithmetic, firm identity, regions, gap detection). A wrong
judgement is a prompt experiment to fix; a wrong lookup is one line of YAML.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

candidates = json.loads((ROOT / "data" / "candidates.json").read_text())
pd.DataFrame([{
    "candidate": c["display_name"],
    "region": c["region"],
    "approach": c["approach_family"],
    "yrs investing": c["years_investment_experience"],
    "sectors": ", ".join(c["sectors"]),
    "firm": c["current_firm"],
    "parse confidence": f'{c["quality"]["score"]} ({c["quality"]["band"]})',
} for c in candidates])

,candidate,region,approach,yrs investing,sectors,firm,parse confidence
0,Chen Li (Alex),APAC,systematic_quant,2.2,"technology, healthcare",NaN,0.96 (high)
1,MARINA SILVA COSTA,Europe,fundamental,5.0,"healthcare, consumer, media_telecom",Vanguard Group,0.88 (high)
2,Marcus Chen-Rodriguez,US,fundamental,9.2,healthcare,Coatue Management,0.94 (high)
3,Michael Rodriguez,US,fundamental,7.0,"technology, media_telecom",Fidelity Asset Management,0.9 (high)
4,Omar El-Hassan,Europe,systematic_quant,0.0,macro_rates_fx,BNP Paribas,0.88 (high)
5,Priya Nakamura,APAC,fundamental,12.7,healthcare,ICICI Securities,0.66 (medium)
6,Ryan Patel,US,fundamental,6.3,"consumer, technology, healthcare",Meridian Capital Partners,0.84 (high)
7,Vikram Shah,US,fundamental,7.5,"technology, media_telecom",Cinctive Capital Management,0.98 (high)
8,Viktor Sharat,APAC,fundamental,9.8,"healthcare, energy, industrials",NaN,0.45 (low)
9,Dr. Zara Al-Rashid,APAC,fundamental,10.6,healthcare,Meridian Research Partners,0.58 (medium)


## 2 · Reading the documents (where most pipelines silently fail)

The largest source of error in a resume pipeline is not the model — it is
**losing content before the model ever sees it**. Three real failures in this
corpus, all caught because the extraction layer reports diagnostics instead
of passing whatever it got:

- **`Viktor_Sharat.docx` keeps its name, degree and section headings in
  floating text boxes** — elements `python-docx` does not read at all. Naive
  extraction returned 503 characters and a resume with apparently no name;
  walking the raw XML for `w:txbxContent` recovered 3,100.
- **`Zara_AlRashid.docx` keeps its work history in tables** (2,848 → 3,401
  chars once tables are read in document order).
- **`Omar_ElHassan.pdf` has a broken ToUnicode map**: ligatures decode to
  U+FFFD, so "Quantitative" arrives as "Quan�ta�ve". 35 damaged tokens were
  repaired by rule (word-shape patterns — an early `ffi` rule corrupted
  "Statistics" and was removed after the model itself reported the damage);
  2 unrecoverable ones are *reported*, not guessed.
- Two-column PDFs are split by **line-start bimodality** before reading —
  a first histogram-based version failed on pages where the main column
  spans the width, and a second version false-positived on three centred
  header lines; the shipped rule requires the right cluster to be at least
  4 lines and 15% of the page.

Per-document diagnostics from the committed extraction log:

In [2]:
log = pd.read_csv(ROOT / "data" / "extraction_log.csv")
log

,source_file,file_type,char_count,table_count,table_share,textbox_count,textbox_chars,page_count,multi_column_detected,ligature_repairs,replacement_chars_remaining,warnings
0,Chen_Li_Alex.docx,docx,4270,0,0.000,0,0,0,False,0,0,NaN
1,MARINA_SILVA_COSTA.docx,docx,3499,0,0.000,0,0,0,False,0,0,NaN
2,Marcus_ChenRodriguez_Resume.docx,docx,3407,0,0.000,0,0,0,False,0,0,NaN
3,Michael_Rodriguez_CFA.docx,docx,3608,1,0.034,0,0,0,False,0,0,NaN
4,Omar_ElHassan_202405.pdf,pdf,1639,0,0.000,0,0,1,True,35,2,Multi-column layout detected - columns were se...
5,Priya_Nakamura_sellside_healthcare_RLTM.docx,docx,5188,1,0.049,0,0,0,False,0,0,NaN
6,RYAN_PATEL__Resume.pdf,pdf,4953,0,0.000,0,0,2,False,0,0,NaN
7,Vikram_Shah.docx,docx,3880,0,0.000,0,0,0,False,0,0,NaN
8,Viktor_Sharat.docx,docx,3100,2,0.824,3,40,0,False,0,0,3 floating text box(es) recovered and hoisted ...
9,Zara_AlRashid.docx,docx,3401,7,0.164,0,0,0,False,0,0,NaN


## 3 · Structured extraction by the model

`src/parse.py` calls the Anthropic API with the Pydantic schema in
`src/schema.py` passed as a **tool definition**, so the model must return an
object of exactly that shape — no free-text JSON to repair. Three guards sit
on top:

- **Corrective retry**: a validation failure is sent back once *with the
  specific errors attached*, not blindly re-rolled.
- **Verbatim-quote verification**: every evidence quote is checked to appear
  in the source text (whitespace/punctuation forgiven — our own extraction
  reflows lines — words never). Unverified quotes reduce the record's
  confidence score.
- **Content-hash cache** keyed on (text, model, schema, system prompt):
  re-running the pipeline after a knowledge-base change costs $0.00. The
  key *includes the prompt* because an early version silently replayed
  pre-fix results after a prompt fix.

One prompt lesson worth recording: the filename was originally included in
the prompt, and the model used it for the candidate's name on some runs and
refused on others. **Ambiguous instructions produce non-deterministic
output — the bug was in the prompt, not the model.** The filename is now
withheld; a deterministic `name_from_filename()` fallback applies only when
the document itself states no name, and flags the record.

Parsing all 10 resumes cost **≈ $0.80** (claude-sonnet-5); the public app
calls no model and needs no key.

In [3]:
from parse import SYSTEM_PROMPT
print(SYSTEM_PROMPT)

You extract structured data from investment-industry resumes for a hedge fund business development team.

Rules that matter more than completeness:

1. Transcribe, do not embellish. If the resume does not state something, return null. A null is useful; an invented value is a liability.

2. Evidence must be VERBATIM. Every `evidence` field must be a substring of the resume text, copied exactly. Never paraphrase, summarise or reconstruct a quote. If you cannot find supporting text, return an empty string and set confidence to "low".

3. Classify by substance, not vocabulary. What someone DID outranks what they called it. "Python" in a skills list is not evidence of systematic investing; a backtested factor model is.

4. Section headings lie. Some resumes file work history under "ACADEMIC PROFILE" or "KEY PROJECTS". An entry naming an employer, a role and a duration is a position regardless of the heading above it.

5. You are given the document text only, never its filename. If the resum

In [4]:
# A parsed record, abbreviated: the approach/sector judgements carry their
# evidence, and the quotes are verified verbatim against the source text.
ryan = next(c for c in candidates if c["display_name"] == "Ryan Patel")
e = ryan["extraction"]
{
    "investment_approach": e["investment_approach"],
    "primary_sectors": e["primary_sectors"][:2],
    "positions[0]": {k: e["positions"][0][k] for k in
                     ("firm", "title", "start_date", "end_date",
                      "employment_type", "is_investment_role")},
}

{'investment_approach': {'value': 'fundamental',
  'keywords': ['fundamental analyses',
   'management meetings',
   'company diligence'],
  'evidence': 'Presented new investment ideas based on fundamental analyses, primary research, industry events/conferences, sector expert consultations, and company management meetings',
  'confidence': 'high'},
 'primary_sectors': [{'value': 'consumer',
   'keywords': ['Consumer', 'TMT', 'portfolio'],
   'evidence': 'Investment Analyst, Consumer & TMT – North53 Capital',
   'confidence': 'high'},
  {'value': 'technology',
   'keywords': ['Technology portfolio', 'Consumer'],
   'evidence': 'Principal analyst responsible for managing the Consumer and Technology portfolio within the $4.2bn gross portfolio for Vertex Capital',
   'confidence': 'high'}],
 'positions[0]': {'firm': 'Meridian Capital Partners',
  'title': 'Investment Professional, Generalist – Soft Catalyst & Fundamental Long/Short',
  'start_date': '2023-03',
  'end_date': None,
  'employ

## 4 · The knowledge base supplies what no model knows

`knowledge/` holds curated domain facts a language model cannot be trusted
to supply (and does not signal when it is guessing): firm identities and
pod-to-platform lineage, region and sector taxonomies, credential
expansions, requisitions, and the human flag-triage file.

Three behaviours worth demonstrating live:

In [5]:
from knowledge_base import KnowledgeBase, years_of_experience
kb = KnowledgeBase.load(ROOT / "knowledge")

# 1. Refusal to guess: four unrelated firms here begin with "Meridian".
#    A substring matcher would silently relocate a candidate to the wrong
#    continent; this one reports ambiguity instead of resolving.
print("resolve('Meridian')      ->", kb.resolve_firm("Meridian").method)

# 2. Pod-to-platform lineage: the resume names only the pod; the platform
#    exists only here. This is what lets the app surface "previously at
#    Millennium" for Ryan Patel.
print("lineage('North53 Capital') ->", kb.platform_lineage("North53 Capital"))

# 3. Tenure is date arithmetic, never model output. Overlapping positions
#    merge; internships and student societies are excluded -- counting them
#    added four years to one candidate in this pool.
overlap = [
    {"start_date": "2020-01", "end_date": "2022-01", "is_current": False,
     "employment_type": "professional", "is_investment_role": True},
    {"start_date": "2021-01", "end_date": "2023-01", "is_current": False,
     "employment_type": "professional", "is_investment_role": True},
]
print("overlapping 2y+2y roles  ->", years_of_experience(overlap), "years")

resolve('Meridian')      -> ambiguous
lineage('North53 Capital') -> ['Millennium Management']
overlapping 2y+2y roles  -> 3.0 years


A later addition in the same spirit: **human flag triage**
(`knowledge/flag_review.yaml`). The model reports everything it notices;
a human reviewer marked several observations benign (a summer internship
inside an MBA, non-US number formatting from a non-US candidate). Those
decisions are *knowledge*: each is recorded with its reasoning, downgrades
the flag to an unscored note, and is never a silent deletion — triage stays
auditable and reversible.

## 5 · Requisition matching: eligibility, then rank

All four shipped requisitions are transcribed from **real postings** — three
Millennium (REQ-27950, REQ-25042, REQ-29449) and one Point72 — rather than
written to fit the data. That matters: a requisition invented alongside the
scoring logic can only confirm itself. Transcription was faithful even where
inconvenient: the Mumbai role's "healthcare preferred but not mandatory"
means sector is *not* a hard constraint there; the Origination role states
no years band, so it has none.

**Hard constraints disqualify** (region, approach family, sector-any,
experience band). **Soft signals rank** the survivors — a weighted blend of
sector fit, requirement-text similarity, skills, firm type, coverage depth,
credentials, platform lineage and buy-side experience, with the weights in
`knowledge/requisitions.yaml` where they can be argued about. Requirement
similarity is a pluggable backend; the default combines lexical overlap
with a **curated concept map** (a requisition says "catalysts" where a
resume says "earnings events"). Neural sentence embeddings drop in
unchanged when the corpus outgrows curation (~low tens of thousands of
documents); at 200 candidate sentences the map performs comparably, adds no
500MB dependency to a free-tier deployment, and is auditable — a bad match
is fixed by editing a line of YAML.

Near misses — candidates failing **exactly one** hard requirement — are
listed separately with the failed requirement and both numbers named.
Failing two or more means a different person; padding lists with them is
the behaviour this system exists to avoid.

In [6]:
from match import Requisitions, match_all
store = Requisitions.load(ROOT / "knowledge")

rows = []
for spec in store.items:
    exact, near = match_all(candidates, spec, store=store)
    rows.append({
        "requisition": spec["title"],
        "source": spec.get("source", ""),
        "qualify": len(exact),
        "one gap away": len(near),
        "top match": (f'{exact[0].display_name} ({exact[0].soft_score:.0%})'
                      if exact else "—"),
    })
pd.DataFrame(rows)

,requisition,source,qualify,one gap away,top match
0,Equity Analyst - US Healthcare Therapeutics,Point72 posting (real),2,6,Ryan Patel (63%)
1,US Healthcare Origination Associate,Millennium posting (real),2,7,Ryan Patel (56%)
2,"Research Analyst, Healthcare (Mumbai)",Millennium posting (real),0,4,—
3,"Quantitative Analyst, Quantitative Strategies",Millennium posting (real),0,1,—


In [7]:
# The result the design is proudest of: an honest zero. Against the Mumbai
# posting's 4-5 year band, nobody qualifies -- and instead of a confidently
# ranked list, the system names the single gap for each near miss.
spec = store.get("mlm_mumbai_healthcare_research")
exact, near = match_all(candidates, spec, store=store)
pd.DataFrame([{
    "candidate": r.display_name,
    "fit (soft)": f"{r.soft_score:.0%}",
    "the one gap": f"{r.failed_hard[0].label}: has {r.failed_hard[0].found}, "
                   f"role needs {r.failed_hard[0].required}",
} for r in near])

,candidate,fit (soft),the one gap
0,Priya Nakamura,60%,"Investment experience: has 12.7 years, role ne..."
1,Dr. Zara Al-Rashid,58%,"Investment experience: has 10.6 years, role ne..."
2,Viktor Sharat,57%,"Investment experience: has 9.8 years, role nee..."
3,MARINA SILVA COSTA,54%,"Region: has Europe, role needs APAC"


## 6 · The bug that justifies the evidence rule

Mid-project, every requirement similarity score came out around 0.75 and the
ranking looked plausible. The aggregate numbers hid the cause completely;
**one on-screen evidence quote exposed it** — the scorer offered
*"Biomodeller Trainee BioAnalytics Research India Ltd."* as proof of
*"fundamental research on India equity"*.

The concept map contained `r` (the R language), matched as a raw substring —
and `"r" in sentence` is true of nearly every English sentence, so every
requirement scored on its concept term alone. The fix was word-boundary
regex matching plus short-sentence damping. The same substring lesson
resurfaced twice more (a title-hint rule read the "intern" inside
"Consumer **Intern**et"; an early flag matcher over-merged) — which is why
the shipped code matches tokens, never substrings, everywhere.

The design conclusion: **a scoring system whose every claim is quoted is a
scoring system whose bugs are visible.** That is why evidence is a hard
requirement of the schema, not decoration.

## 7 · Accuracy against blind human judgment

Percentages first, method second — but the method is the point:

- A reviewer (the author, acting as the BD screener) labelled **all 40
  candidate-role pairs** blind: the candidate's facts were visible, the
  system's verdict never was. Y / N / borderline; borderline is excluded
  from counts rather than forced into a bucket.
- The system's shortlist is its **exact matches only** — near misses do not
  count for it.

In [8]:
from evaluate import evaluate
ev = evaluate()
per = pd.DataFrame(ev["per_role"])[
    ["role", "precision", "recall", "agreement", "tp", "fp", "fn", "tn"]]
o = ev["overall"]
print(f'OVERALL  precision {o["precision"]:.0%}  recall {o["recall"]:.0%}  '
      f'agreement {o["agreement"]:.0%}  (n={o["judged"]} judged, '
      f'{len(ev["borderline"])} borderline set aside)')
per

OVERALL  precision 100%  recall 57%  agreement 92%  (n=36 judged, 4 borderline set aside)


,role,precision,recall,agreement,tp,fp,fn,tn
0,Equity Analyst - US Healthcare Therapeutics,1.0,1.0,1.0,2,0,0,8
1,US Healthcare Origination Associate,1.0,1.0,1.0,2,0,0,8
2,"Research Analyst, Healthcare (Mumbai)",NaN,0.0,0.7,0,0,3,7
3,"Quantitative Analyst, Quantitative Strategies",NaN,NaN,1.0,0,0,0,6


In [9]:
pd.DataFrame(ev["disagreements"])

,role,candidate,kind,system_reason
0,"Research Analyst, Healthcare (Mumbai)",Priya Nakamura,false_negative,"Investment experience: has 12.7 years, role ne..."
1,"Research Analyst, Healthcare (Mumbai)",Viktor Sharat,false_negative,"Investment experience: has 9.8 years, role nee..."
2,"Research Analyst, Healthcare (Mumbai)",Zara Al-Rashid,false_negative,"Investment experience: has 10.6 years, role ne..."


**Reading the disagreements.** Precision is perfect on this pool — the
system never shortlisted anyone the reviewer would reject. Every miss has a
single cause: the Mumbai posting's 4–5 year band excludes three APAC
healthcare analysts at 9.8–12.7 years, all of whom the reviewer shortlists.
Nothing comparable happened with region — all nine region mismatches were
labelled *no* — so the reviewer treats geography as genuinely hard while
treating **over-qualification as negotiable**, a distinction the posting's
text does not make. The borderline labels cluster the same way, on the
quant seat's approach constraint.

The reviewer's own account sharpens it further: the widening was
**supply-driven** — with ten candidates she stretches the band; with ten
thousand she would not. Shortlisting standards are elastic to pool depth,
which no fixed threshold can encode. The design response is not to soften
the band but to keep it transcribed and make widening a *visible,
per-search decision*: all three missed candidates sit at the top of the
one-gap-away list with the band named on their row, and the empty state
computes what each widening would admit.

At n=40 the percentages measure nothing statistically. What the exercise
measures is **which rule diverges from practitioner judgment** — and it
found exactly one. The same harness (`src/evaluate.py` + a labelling sheet)
is the regression suite for any future scoring change.

## 8 · The application

**Live: https://m-case-study-jasmine.streamlit.app/** (public; calls no
model — the deployment serves precomputed data, so no API key exists in the
app to leak or spend).

![Candidates view](docs/app_candidates.png)

The interface went through ~15 review rounds with two reviewers (a BD-user
perspective and a UI designer). The decisions that survived:

- **One screen, three zones**: role (eligibility) → advanced filters
  (refinement) → results with profile. When a role fixes a dimension, the
  sidebar filter for it *disappears* — a second control over the same axis,
  even disabled, read as a second authority.
- **Qualify vs One-gap-away** as separate collapsible groups; rows are one
  line each (name, Fit, two facts, quality tag) because the *why* lives one
  glance right, in the Fit panel.
- **Colour grammar**: navy is chrome, green means "clears every hard
  requirement", bronze means gap/caveat, and every informational tag is one
  neutral pill. Verdict facts may be *tags with tinted backgrounds*;
  coloured *text* was reviewed out.
- **Evidence on demand**: profile shows keywords and per-dimension
  contributions ("What counted"); verbatim quotes live one expander deeper.
- **Outreach draft**: the step after "this candidate fits" is always
  "someone writes to them" — one click assembles a summary whose every
  claim quotes the resume, because a sourcing mail earns replies by proving
  someone actually read it.

![Shortlist table](docs/app_table.png)

## 9 · Designed for ten, priced for a hundred thousand

Every scale-sensitive choice has a stated threshold and a successor:

| Component | At 10 resumes (shipped) | At ~100k resumes |
|---|---|---|
| Parsing | sequential calls, content-hash cache | queue + batch API; ≈ $0.08/resume ⇒ ≈ $8k for 100k, incremental thereafter (cache means re-enrichment is free) |
| Requirement similarity | curated concept map (auditable YAML) | neural sentence embeddings via the pluggable backend in `src/match.py`; switch point ≈ low tens of thousands of documents |
| Search | in-memory filtering | SQLite FTS / OpenSearch index + pgvector for semantic recall |
| Entity resolution | curated `firms.yaml` | the same guarded resolver over a licensed firm graph (e.g. FactSet entities), still refusing ambiguous matches |
| Quality & eval | per-record confidence; 40-pair human eval | sampled human labelling as a monitored metric; drift alarms on parse-confidence distribution |
| Serving | Streamlit Community Cloud | internal deployment; JD upload wired to the parsing service with managed keys (deliberately absent from the public app) |

The **evaluation harness is the keystone for scale**: any threshold change,
prompt change or backend swap re-runs against the labelled pairs before it
ships.

## 10 · If I had more time (and the features I chose not to fake)

- **Referral & CRM metadata.** "Referred by" is not in any resume — it is
  ATS-side data. The record schema gets a `referral` field (default
  unknown) and a sidebar facet; kept out now rather than invented.
- **Verified performance.** Self-reported AUM and returns are shown today
  as quoted, clearly-labelled statements only — they are unverifiable and
  most resumes omit them, so as a *ranking* signal they would reward
  disclosure habits, not ability. With a licensed data source (fund
  filings), book size and track record become real, scoreable fields.
- **Internal sourcing corpus + RAG.** Meeting notes and call summaries
  would let free-text questions ("who impressed us on biotech last year?")
  join the structured search — with governance caveats: retrieval scope,
  permissions, and the same evidence-quote discipline.
- **In-app labelling.** The ground-truth workflow (§7) moves into the app:
  pick a role, label candidates, accuracy recomputes — evaluation as a
  habit, not an event.
- **JD upload in a governed deployment.** Parsing a pasted JD is the same
  extraction problem as parsing a resume and the code path exists; it is
  withheld from the public app only because it requires a live key in a
  public page. An internal deployment removes that constraint.
- **Knowledge graph.** `firms.yaml` is already a small graph (firm → parent
  → platform); at scale it becomes queryable lineage ("everyone two hops
  from a Millennium pod").

## Appendix · Reproducing everything

```bash
pip install -r requirements.txt
streamlit run app.py                 # the app, against committed data

pip install -r requirements-dev.txt
python -m pytest tests/ -q           # 11 regression tests, no API key needed
python src/evaluate.py               # accuracy vs the human labels

# full rebuild from raw resumes (needs resumes in data/resumes/ and an
# ANTHROPIC_API_KEY in .env; cached, so re-runs are free):
python src/build_dataset.py
python tools/build_notebook.py       # re-executes this notebook
```